# Phase 6: Retrieval Pipeline

Evaluate Dense, Sparse (BM25), and Hybrid (RRF) retrieval methods across 3 chunking strategies.
Uses `qa_pairs_filtered.parquet` for evaluation.

In [1]:
import os, sys, subprocess
from pathlib import Path

# Detect environment
def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    REPO_ROOT = Path('/content/rag-vn-finance')
    if not REPO_ROOT.exists():
        print("Đang tải mã nguồn và cài đặt thư viện lần đầu...")
        subprocess.run(['git', 'clone', 'https://github.com/thong7d/rag-vn-finance.git', str(REPO_ROOT)])
        
        req_path = REPO_ROOT / 'requirements.txt'
        if req_path.exists():
            os.system(f'pip install -r "{req_path}" -q')
            
        print("Cài đặt hoàn tất. Đang tự động khởi động lại Kernel để nạp thư viện lõi (Numpy/Torch)...")
        os.kill(os.getpid(), 9) # Tự động ngắt tiến trình để ép Colab khởi động lại RAM
    else:
        print("Mã nguồn đã tồn tại. Bỏ qua cài đặt...")
else:
    REPO_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())

print(f"Project root: {REPO_ROOT}")
assert REPO_ROOT.exists(), f"Project root not found: {REPO_ROOT}"

src_path = str(REPO_ROOT)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Load file .env từ Google Drive vào hệ thống Colab
if IN_COLAB:
    from dotenv import load_dotenv
    load_dotenv('/content/drive/MyDrive/rag-vn-finance/.env') 

print("\nColab setup complete.")

Environment: Google Colab
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mã nguồn đã tồn tại. Bỏ qua cài đặt...
Project root: /content/rag-vn-finance

Colab setup complete.


## 1. Environment Setup & Model Loading
Import necessary libraries, load project configurations from `config.yaml`, and initialize the SentenceTransformer model used for encoding user queries.

In [2]:
import json
import pandas as pd
import faiss
from tqdm import tqdm  # Đã đổi từ tqdm.notebook sang tqdm chuẩn để tránh lỗi hiển thị trên Colab
from sentence_transformers import SentenceTransformer
import torch

from src.utils import load_config, resolve_path, ensure_dir
from src.indexing import load_bm25_index
from src.retrieval import DenseRetriever, SparseRetriever, HybridRetriever, calculate_metrics

# Load config
config = load_config()

# Load Model
model_name = config['embedding']['model_name']
device = config['embedding'].get('device', 'cpu')
if device == 'auto':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer(model_name, device=device)
print(f"Loaded {model_name} on {model.device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Loaded intfloat/multilingual-e5-large on cuda:0


## 2. Load Evaluation Data
Load the synthetic QA pairs generated in Phase 5. We use the full `qa_pairs_filtered.parquet` dataset to evaluate our retrieval methods.

In [3]:
qa_dir = resolve_path(config['synthetic_qa'], 'output_dir')
qa_path = os.path.join(qa_dir, 'qa_pairs_filtered.parquet')

if not os.path.exists(qa_path):
    raise FileNotFoundError(f"{qa_path} not found. Ensure Phase 5 is completed.")

df_qa = pd.read_parquet(qa_path)
print(f"Loaded {len(df_qa)} QA pairs for evaluation.")

Loaded 983 QA pairs for evaluation.


## 3. Retrieval Evaluation Loop
Iterate through all 3 chunking strategies (`fixed_size`, `sentence_aware`, `article_level`). For each strategy:
1. Load the corresponding FAISS and BM25 indices.
2. Instantiate Dense, Sparse, and Hybrid retrievers.
3. Run all evaluation queries and compute `Precision@K`, `Recall@K`, `MRR`, and `NDCG@10`.
4. Aggregate the metrics.

In [4]:
strategies = config['chunking']['strategies']
top_k_dense = config['retrieval']['top_k_dense']
top_k_sparse = config['retrieval']['top_k_sparse']
top_k_hybrid = config['retrieval']['top_k_hybrid']
rrf_k = config['retrieval']['rrf_k']

index_base_dir = resolve_path(config['indexing'], 'output_dir')
bm25_base_dir = resolve_path(config['indexing'], 'bm25_dir')

results = []

for strategy in strategies:
    print(f"\n{'='*40}\nEvaluating Strategy: {strategy}\n{'='*40}")
    
    # --- Load Dense Index ---
    faiss_path = os.path.join(index_base_dir, strategy, "index.faiss")
    chunk_ids_path = os.path.join(index_base_dir, strategy, "chunk_ids.json")
    
    if not os.path.exists(faiss_path):
        print(f"Missing Dense index for {strategy}. Skipping.")
        continue
        
    faiss_index = faiss.read_index(faiss_path)
    with open(chunk_ids_path, 'r', encoding='utf-8') as f:
        dense_chunk_ids = json.load(f)
        
    dense_retriever = DenseRetriever(faiss_index, dense_chunk_ids, model)
    
    # --- Load Sparse Index ---
    try:
        bm25_index, sparse_chunk_ids = load_bm25_index(bm25_base_dir, strategy)
    except FileNotFoundError:
        print(f"Missing Sparse index for {strategy}. Skipping.")
        continue
        
    sparse_retriever = SparseRetriever(bm25_index, sparse_chunk_ids)
    
    # --- Hybrid Retriever ---
    hybrid_retriever = HybridRetriever(dense_retriever, sparse_retriever, rrf_k=rrf_k)
    
    # --- Evaluate ---
    strategy_metrics = {'Dense': [], 'Sparse': [], 'Hybrid': []}
    
    print("Bắt đầu chạy vòng lặp đánh giá (vui lòng đợi vài phút cho mỗi strategy)...")
    
    # Progress bar over the evaluation dataset
    for _, row in tqdm(df_qa.iterrows(), total=len(df_qa), desc=f"Evaluating {strategy}"):
        query = row['question']
        ground_truth_doc_id = row['doc_id']
        
        # Dense
        dense_res = dense_retriever.retrieve(query, top_k=top_k_dense)
        dense_ids = [cid for cid, _ in dense_res]
        strategy_metrics['Dense'].append(calculate_metrics(dense_ids, ground_truth_doc_id, k=top_k_dense))
        
        # Sparse
        sparse_res = sparse_retriever.retrieve(query, top_k=top_k_sparse)
        sparse_ids = [cid for cid, _ in sparse_res]
        strategy_metrics['Sparse'].append(calculate_metrics(sparse_ids, ground_truth_doc_id, k=top_k_sparse))
        
        # Hybrid
        hybrid_res = hybrid_retriever.retrieve(query, top_k=top_k_hybrid)
        hybrid_ids = [cid for cid, _ in hybrid_res]
        strategy_metrics['Hybrid'].append(calculate_metrics(hybrid_ids, ground_truth_doc_id, k=top_k_hybrid))
        
    # --- Aggregate and Store Results ---
    for method, metrics_list in strategy_metrics.items():
        df_m = pd.DataFrame(metrics_list)
        avg_metrics = df_m.mean().to_dict()
        
        res_row = {
            'Strategy': strategy,
            'Method': method
        }
        res_row.update(avg_metrics)
        results.append(res_row)
        
        # Format metrics for printing
        metrics_str = ", ".join([f"{k}: {v:.4f}" for k, v in avg_metrics.items()])
        print(f"{method:10} -> {metrics_str}")
        
    # --- Checkpoint after each strategy ---
    df_results = pd.DataFrame(results)
    eval_dir = resolve_path(config['evaluation'], 'output_dir')
    ensure_dir(eval_dir)
    out_path = os.path.join(eval_dir, "retrieval_benchmark.csv")
    df_results.to_csv(out_path, index=False)
    print(f"\n[CHECKPOINT] Đã lưu kết quả của {strategy} vào {out_path}")



Evaluating Strategy: fixed_size


[2026-05-13 08:35:48] [INFO] src.indexing: [fixed_size] BM25 index loaded — 45,764 chunks, avgdl=206.0
INFO:src.indexing:[fixed_size] BM25 index loaded — 45,764 chunks, avgdl=206.0


Bắt đầu chạy vòng lặp đánh giá (vui lòng đợi vài phút cho mỗi strategy)...


Evaluating fixed_size: 100%|██████████| 983/983 [16:27<00:00,  1.00s/it]


Dense      -> Precision@10: 0.3139, Recall@10: 0.9278, MRR: 0.8253, NDCG@10: 0.8282
Sparse     -> Precision@10: 0.2399, Recall@10: 0.9156, MRR: 0.7917, NDCG@10: 0.7923
Hybrid     -> Precision@10: 0.3066, Recall@10: 0.9512, MRR: 0.8554, NDCG@10: 0.8403

[CHECKPOINT] Đã lưu kết quả của fixed_size vào /content/drive/MyDrive/rag-vn-finance/evaluation/retrieval_benchmark.csv

Evaluating Strategy: sentence_aware


...

Dense      -> Precision@10: 0.2803, Recall@10: 0.9451, MRR: 0.8351, NDCG@10: 0.8232
Sparse     -> Precision@10: 0.2168, Recall@10: 0.8901, MRR: 0.7391, NDCG@10: 0.7459
Hybrid     -> Precision@10: 0.2720, Recall@10: 0.9542, MRR: 0.8410, NDCG@10: 0.8266

[CHECKPOINT] Đã lưu kết quả của sentence_aware vào /content/drive/MyDrive/rag-vn-finance/evaluation/retrieval_benchmark.csv

Evaluating Strategy: article_level


[2026-05-13 09:18:42] [INFO] src.indexing: [article_level] BM25 index loaded — 9,999 chunks, avgdl=404.6
INFO:src.indexing:[article_level] BM25 index loaded — 9,999 chunks, avgdl=404.6


Bắt đầu chạy vòng lặp đánh giá (vui lòng đợi vài phút cho mỗi strategy)...


Evaluating article_level: 100%|██████████| 983/983 [04:21<00:00,  3.76it/s]

Dense      -> Precision@10: 0.0945, Recall@10: 0.9451, MRR: 0.8303, NDCG@10: 0.8585
Sparse     -> Precision@10: 0.0896, Recall@10: 0.8962, MRR: 0.7643, NDCG@10: 0.7965
Hybrid     -> Precision@10: 0.0953, Recall@10: 0.9532, MRR: 0.8417, NDCG@10: 0.8693

[CHECKPOINT] Đã lưu kết quả của article_level vào /content/drive/MyDrive/rag-vn-finance/evaluation/retrieval_benchmark.csv


## 4. Save Benchmark Results
Export the final evaluation results to `evaluation/retrieval_benchmark.csv` for use in subsequent phases or reporting.

In [6]:
if results:
    df_results = pd.DataFrame(results)
    eval_dir = resolve_path(config['evaluation'], 'output_dir')
    ensure_dir(eval_dir)
    
    out_path = os.path.join(eval_dir, "retrieval_benchmark.csv")
    df_results.to_csv(out_path, index=False)
    print(f"\nHoàn thành! Đã lưu kết quả cuối cùng vào {out_path}")
    display(df_results)
else:
    print("No results to save. Ensure indexes and QA data are present.")


Hoàn thành! Đã lưu kết quả cuối cùng vào /content/drive/MyDrive/rag-vn-finance/evaluation/retrieval_benchmark.csv


,Strategy,Method,Precision@10,Recall@10,MRR,NDCG@10
0,fixed_size,Dense,0.313937,0.927772,0.825308,0.828225
1,fixed_size,Sparse,0.239878,0.915565,0.791709,0.792334
2,fixed_size,Hybrid,0.306612,0.951170,0.855406,0.840344
3,sentence_aware,Dense,0.280264,0.945066,0.835089,0.823212
4,sentence_aware,Sparse,0.216785,0.890132,0.739127,0.745884
5,sentence_aware,Hybrid,0.272024,0.954222,0.841002,0.826590
6,article_level,Dense,0.094507,0.945066,0.830300,0.858458
7,article_level,Sparse,0.089624,0.896236,0.764255,0.796530
8,article_level,Hybrid,0.095320,0.953204,0.841713,0.869308
